In [1]:
import wrds
import pandas as pd
from influxdb_client_3 import InfluxDBClient3, Point, WriteOptions, write_client_options

In [2]:
wrds_db = wrds.Connection()

Enter your WRDS username [alanxw]: ryan_eng
Enter your password: ········


WRDS recommends setting up a .pgpass file.


Create .pgpass file now [y/n]?:  y


Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [3]:
def success(self, data: str):
    print(f"Successfully wrote batch: data")

def errcolumnsf, data: str, exception):
    print(f"Failed writing batch: config: {self}, data: {data} due: {exception}")

def retry(self, data: str, exception):
    print(f"Failed retry writing batch: config: {self}, data: {data} retry: {exception}")

# Configure options for batch writing.
write_options = WriteOptions(batch_size=10_000,
                                    flush_interval=10_000,
                                    jitter_interval=2_000,
                                    retry_interval=5_000,
                                    max_retries=5,
                                    max_retry_delay=30_000,
                                    exponential_base=2)

# Create an options dict that sets callbacks and WriteOptions.
wco = write_client_options(success_callback=success,
                          error_callback=error,
                          retry_callback=retry,
                          write_options=write_options)

client = InfluxDBClient3(token=f"apiv3_hoBjjFB7ZYYtYy33a_Ui5VTjsWG848623y8Mlf_a_2gVU86YB8gwVjJVYRJXaTUWnS4desa23-y7DcXa4qxmIg", 
                         host=f"http://localhost:8181", database=f"jkp_factors", write_client_options=wco)

In [47]:
schema = wrds_db.get_table(library="contrib", table="global_factor", obs=0)
schema.columns

Index(['permno', 'permco', 'gvkey', 'iid', 'id', 'date', 'excntry', 'eom',
       'source_crsp', 'size_grp',
       ...
       'bidaskhl_21d', 'rvolhl_21d', 'corr_1260d', 'betabab_1260d',
       'rmax5_rvol_21d', 'age', 'qmj', 'qmj_prof', 'qmj_growth', 'qmj_safety'],
      dtype='object', length=443)

In [48]:
characteristics = ["ret_exc"]

sql_query= f"""SELECT id, eom, gvkey, permno, size_grp, {' ,'.join(characteristics)}
FROM contrib.global_factor
WHERE common=1 and exch_main=1 and primary_sec=1 and obs_main=1 and
excntry='USA'
"""
usa_data = wrds_db.raw_sql(sql_query)

In [49]:
import numpy as np

short_data = usa_data.copy().set_index("eom")
short_data["size_grp"] = short_data["size_grp"].fillna("none")
short_data["permno"] = short_data["permno"].astype("string").fillna("none")
short_data["gvkey"] = short_data["gvkey"].fillna("none")

short_data

,id,gvkey,permno,size_grp,ret_exc
eom,,,,,
1962-02-28,101127101.0,011271,none,none,<NA>
1962-01-31,100112401.0,001124,none,none,<NA>
1962-01-31,100115301.0,001153,none,none,<NA>
1962-01-31,100115701.0,001157,none,mega,<NA>
1962-01-31,100117701.0,001177,none,none,<NA>
...,...,...,...,...,...
2024-08-31,93436.0,184996,93436.0,mega,-0.081809
2024-09-30,93436.0,184996,93436.0,mega,0.217323
2024-10-31,93436.0,184996,93436.0,mega,-0.048932


In [50]:
client.write(short_data, 
             data_frame_measurement_name="characteristics", 
             data_frame_tag_columns=["size_grp", "gvkey", "permno"])

In [8]:
factor_returns = pd.read_csv("factor-returns.csv")
factor_returns = factor_returns.set_index("date")
factor_returns

,location,name,freq,weighting,n_factors,ret
date,,,,,,
1946-01-31,usa,accruals,monthly,vw_cap,1,0.018130
1946-02-28,usa,accruals,monthly,vw_cap,1,-0.011203
1946-03-31,usa,accruals,monthly,vw_cap,1,0.043073
1946-04-30,usa,accruals,monthly,vw_cap,1,0.007563
1946-05-31,usa,accruals,monthly,vw_cap,1,0.019484
...,...,...,...,...,...,...
2024-08-31,usa,value,monthly,vw_cap,18,-0.009068
2024-09-30,usa,value,monthly,vw_cap,18,-0.009672
2024-10-31,usa,value,monthly,vw_cap,18,-0.007026


In [9]:
client.write(factor_returns, 
             data_frame_measurement_name="factor_returns", 
             data_frame_tag_columns=["location", "name", "freq","weighting"])

In [15]:
wrds_db.list_tables(library="comp")

['aco_amda',
 'aco_imda',
 'aco_indfnta',
 'aco_indfntq',
 'aco_indfntytd',
 'aco_indsta',
 'aco_indstq',
 'aco_indstytd',
 'aco_notesa',
 'aco_notesq',
 'aco_notessa',
 'aco_notesytd',
 'aco_pnfnda',
 'aco_pnfndq',
 'aco_pnfndytd',
 'aco_pnfnta',
 'aco_pnfntq',
 'aco_pnfntytd',
 'aco_transa',
 'aco_transq',
 'aco_transsa',
 'aco_transytd',
 'adsprate',
 'asec_amda',
 'asec_imda',
 'asec_notesa',
 'asec_notesq',
 'asec_transa',
 'asec_transq',
 'bank_aacctchg',
 'bank_adesind',
 'bank_afnd1',
 'bank_afnd2',
 'bank_afnddc1',
 'bank_afnddc2',
 'bank_afntind',
 'bank_funda',
 'bank_funda_fncd',
 'bank_fundq',
 'bank_fundq_fncd',
 'bank_iacctchg',
 'bank_idesind',
 'bank_ifndq',
 'bank_ifndytd',
 'bank_ifntq',
 'bank_ifntytd',
 'bank_names',
 'bank_namesq',
 'chars',
 'co_aacctchg',
 'co_aaudit',
 'co_acthist',
 'co_adesind',
 'co_adjfact',
 'co_afnd1',
 'co_afnd2',
 'co_afnddc1',
 'co_afnddc2',
 'co_afntind1',
 'co_afntind2',
 'co_ainvval',
 'co_amkt',
 'co_busdescl',
 'co_cotype',
 'co_f

In [25]:
comp_comp = wrds_db.get_table(library="comp", table="company")
comp_keys = comp_comp.gvkey.unique()

In [13]:
wrds_db.list_tables(library="crsp")

['acti',
 'asia',
 'asib',
 'asic',
 'asio',
 'asix',
 'bmdebt',
 'bmheader',
 'bmpaymts',
 'bmquotes',
 'bmyield',
 'bndprt06',
 'bndprt12',
 'bxcalind',
 'bxdlyind',
 'bxmthind',
 'bxquotes',
 'bxyield',
 'cap',
 'ccm_lookup',
 'ccm_qvards',
 'ccmxpf_linktable',
 'ccmxpf_lnkhist',
 'ccmxpf_lnkrng',
 'ccmxpf_lnkused',
 'comphead',
 'comphist',
 'compmaster',
 'contact_info',
 'core',
 'crsp_cik_map',
 'crsp_daily_data',
 'crsp_header',
 'crsp_monthly_data',
 'crsp_names',
 'crsp_portno_map',
 'crsp_ziman_daily_index',
 'crsp_ziman_monthly_index',
 'cs20yr',
 'cs5yr',
 'cs90d',
 'cst_hist',
 'daily_nav',
 'daily_nav_ret',
 'daily_returns',
 'dividends',
 'dport1',
 'dport2',
 'dport3',
 'dport4',
 'dport5',
 'dport6',
 'dport7',
 'dport8',
 'dport9',
 'dsbc',
 'dsbo',
 'dse',
 'dse62',
 'dse62delist',
 'dse62dist',
 'dse62exchdates',
 'dse62names',
 'dse62nasdin',
 'dse62shares',
 'dseall',
 'dseall62',
 'dsedelist',
 'dsedist',
 'dseexchdates',
 'dsenames',
 'dsenasdin',
 'dseshares',

In [17]:
comp_crsp_linker = wrds_db.get_table(library="crsp", table="ccmxpf_lnkhist")

In [26]:
link_keys = comp_crsp_linker.gvkey.unique()

In [45]:
comp_comp[comp_comp.gvkey=="011271"]

,conm,gvkey,add1,add2,add3,add4,addzip,busdesc,cik,city,...,sic,spcindcd,spcseccd,spcsrc,state,stko,weburl,dldte,ipodate,curr_sp500_flag
10141,WALLACE COMPUTER SVCS INC,011271,2275 Cabot Drive,<NA>,<NA>,<NA>,60532-3630,"Wallace Computer Services, Inc. was acquired b...",0000104348,Lisle,...,2761,370,925,<NA>,IL,0,www.wallacecomputers.com,2003-05-16,<NA>,0.0


In [51]:
a = set(comp_keys)
b = set(link_keys)
a-b

{'216279',
 '187412',
 '003389',
 '027296',
 '179759',
 '051419',
 '170919',
 '034323',
 '051595',
 '014948',
 '026174',
 '051499',
 '051840',
 '036062',
 '216290',
 '180140',
 '179370',
 '051474',
 '196555',
 '216221',
 '216450',
 '019250',
 '051529',
 '180570',
 '163429',
 '051227',
 '051498',
 '045801',
 '023063',
 '041244',
 '180608',
 '179762',
 '051463',
 '179243',
 '024457',
 '036358',
 '178284',
 '017094',
 '152208',
 '051594',
 '051438',
 '038492',
 '023056',
 '032966',
 '025306',
 '051546',
 '286873',
 '033296',
 '180623',
 '189057',
 '185644',
 '050982',
 '026560',
 '051162',
 '020613',
 '183308',
 '216454',
 '179377',
 '178133',
 '186757',
 '216250',
 '171102',
 '045354',
 '035713',
 '179538',
 '050795',
 '187050',
 '183872',
 '216469',
 '023571',
 '180239',
 '179989',
 '017053',
 '178396',
 '178589',
 '186495',
 '216247',
 '018744',
 '036141',
 '050912',
 '026331',
 '017259',
 '045547',
 '189538',
 '179760',
 '038600',
 '051303',
 '013885',
 '193755',
 '041355',
 '015138',